In [1]:
# IMDB Sentiment Classification using Deep Neural Network
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models

In [5]:
import pandas as pd

df = pd.read_csv(
    'IMDB_Dataset.csv',
    encoding='latin-1',
    engine='python',
    on_bad_lines='skip'
)

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
negative    24813
positive    24799
Name: count, dtype: int64


In [6]:
# Convert labels: 'positive' → 1, 'negative' → 0
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])     # Encodes text labels to numbers
texts = df['review'].values               # Raw text reviews

In [7]:
# Tokenization — convert words to numbers
max_words = 10000                          # Only top 10,000 words
max_len   = 200                            # Max words per review
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)              # Build vocabulary from text
X = tokenizer.texts_to_sequences(texts)   # Convert text to number sequences
X = pad_sequences(X, maxlen=max_len)      # Pad shorter reviews with 0s

In [8]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [16]:
# Build the neural network
model = models.Sequential([
  # Embedding: converts word indices into dense vectors
  layers.Embedding(max_words, 32, input_length=max_len),
  # Flatten the 2D embedding into 1D for Dense layers
  layers.GlobalAveragePooling1D(),
  layers.Dense(64, activation='relu'),    # Hidden layer
  layers.Dropout(0.5),                     # Randomly drops 50% neurons (prevents overfitting)
  layers.Dense(1, activation='sigmoid')  # Output: probability 0-1 (sigmoid for binary)
])

In [17]:
# Compile model for binary classification
model.compile(
  optimizer='adam',
  loss='binary_crossentropy',            # Loss for binary (0/1) problems
  metrics=['accuracy']                   # We care about % correct
)


In [18]:
model.summary()                            # Print model architecture

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d_1           │ ?                           │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:

# Train the model
history = model.fit(
  X_train, y_train,
  epochs=5,                                # 5 passes (text training is slower)
  batch_size=64,
  validation_split=0.1,
  verbose=1
)

Epoch 1/5
559/559 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7753 - loss: 0.4707 - val_accuracy: 0.8770 - val_loss: 0.3037
Epoch 2/5
559/559 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.8822 - loss: 0.2825 - val_accuracy: 0.8844 - val_loss: 0.2749
Epoch 3/5
559/559 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9063 - loss: 0.2383 - val_accuracy: 0.8871 - val_loss: 0.2700
Epoch 4/5
559/559 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9198 - loss: 0.2109 - val_accuracy: 0.8891 - val_loss: 0.2779
Epoch 5/5
559/559 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9288 - loss: 0.1900 - val_accuracy: 0.8864 - val_loss: 0.2900


In [20]:
# Evaluate on test data
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.2f}%")

Test Accuracy: 88.88%


In [21]:
# Predict sentiment for a new review
sample = ["This movie was absolutely brilliant and wonderful!"]
seq = tokenizer.texts_to_sequences(sample)
seq = pad_sequences(seq, maxlen=max_len)
pred = model.predict(seq)[0][0]
print("Positive" if pred > 0.5 else "Negative", f"({pred:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
Positive (0.91)


In [22]:
model.summary()                            # Print model architecture

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 200, 32)             │         320,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d_1           │ (None, 32)                  │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 966,533 (3.69 MB)

 Trainable params: 322,177 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 644,356 (2.46 MB)